# 03 · Neural network training — AgriTech

Manual backprop, efficiency MLP, then crop-yield GD variants.

Source: `NeuralNetworks_Training_Exercise`

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## Task 1 — equipment cost $f(x)=x^2$
Single linear unit. Compare hand-computed grads with autograd.

In [ ]:
material_amount = torch.tensor([[5.0]])
target_cost = material_amount ** 2  # 25
print('input', material_amount.item(), 'target', target_cost.item())

torch.manual_seed(0)
weight = torch.tensor([[0.4]], requires_grad=True)
bias = torch.tensor([[0.1]], requires_grad=True)

predicted_cost = material_amount * weight + bias
loss = ((predicted_cost - target_cost) ** 2).mean()
print('pred', predicted_cost.item(), 'mse', loss.item())

In [ ]:
# dL/dpred = 2*(pred - target) / n, n=1
dL_dpred = 2 * (predicted_cost.detach() - target_cost)
dpred_dw = material_amount          # ∂(wx+b)/∂w = x
dpred_db = torch.ones_like(bias)    # ∂(wx+b)/∂b = 1
manual_dw = dL_dpred * dpred_dw
manual_db = dL_dpred * dpred_db
print('manual dw', manual_dw.item(), 'manual db', manual_db.item())

loss.backward()
print('autograd dw', weight.grad.item(), 'autograd db', bias.grad.item())
print('dw close?', torch.allclose(manual_dw, weight.grad))
print('db close?', torch.allclose(manual_db, bias.grad))

## Task 2 — production efficiency
$score = (0.4 \cdot maint + 0.6 \cdot train)/5$

In [ ]:
np.random.seed(42)
n = 200
machine_maintenance_hours = np.random.uniform(5, 50, n)
training_hours = np.random.uniform(2, 20, n)
efficiency_score = (0.4 * machine_maintenance_hours + 0.6 * training_hours) / 5.0

X = np.column_stack((machine_maintenance_hours, training_hours))
y = efficiency_score.reshape(-1, 1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
print(X_train.shape, y_train.shape)

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 5),
    nn.ReLU(),
    nn.Linear(5, 1),
    nn.Sigmoid(),
)

def train_model(model, loss_fn, optimizer, X, y, X_val, y_val, epochs):
    train_losses, val_losses = [], []
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            vloss = loss_fn(model(X_val), y_val).item()
        val_losses.append(vloss)
        if (epoch + 1) % 40 == 0 or epoch == 0:
            print(f'Epoch {epoch+1}/{epochs}  train {loss.item():.4f}  val {vloss:.4f}')
    return train_losses, val_losses

epochs = 200
learning_rate = 0.01
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
train_losses, val_losses = train_model(
    model, loss_fn, optimizer, X_train, y_train, X_val, y_val, epochs
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss', linestyle='--')
plt.title('Efficiency net — loss (200 ep, lr=0.01)')
plt.xlabel('Epochs')
plt.ylabel('MSE')
plt.legend()
plt.show()

Bonus: 500 epochs, lr = 0.05. Fresh weights so the two runs stay independent.

In [ ]:
model_fast = nn.Sequential(nn.Linear(2, 5), nn.ReLU(), nn.Linear(5, 1), nn.Sigmoid())
opt_fast = optim.SGD(model_fast.parameters(), lr=0.05)
tl, vl = train_model(model_fast, nn.MSELoss(), opt_fast, X_train, y_train, X_val, y_val, 500)
plt.figure(figsize=(8, 4))
plt.plot(tl, label='Training Loss')
plt.plot(vl, label='Validation Loss', linestyle='--')
plt.title('Bonus — 500 ep, lr=0.05')
plt.xlabel('Epochs')
plt.ylabel('MSE')
plt.legend()
plt.show()

## Task 3 — crop yield: batch vs mini-batch vs SGD

In [ ]:
CSV_NAME = 'agriculture_dataset_codebasics_DL.csv'
RAW = 'https://raw.githubusercontent.com/yash9614/lear_deepintodl/main/NeuralNetworks_Training_Exercise/agriculture_dataset_codebasics_DL.csv'
LOCALS = [CSV_NAME, f'../NeuralNetworks_Training_Exercise/{CSV_NAME}', f'/content/{CSV_NAME}']

def load_csv(raw, locals_, name):
    try:
        return pd.read_csv(raw)
    except Exception:
        pass
    for p in locals_:
        if os.path.exists(p):
            return pd.read_csv(p)
    from google.colab import files
    print('Upload', name)
    up = files.upload()
    return pd.read_csv(next(iter(up)))

agri = load_csv(RAW, LOCALS, CSV_NAME)
print(agri.head())
target_col = [c for c in agri.columns if 'yield' in c.lower()][0]
X = agri.drop(columns=[target_col]).values.astype('float32')
y = agri[target_col].values.astype('float32').reshape(-1, 1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
print(X_train.shape)

In [ ]:
def make_yield_net():
    return nn.Sequential(
        nn.Linear(6, 10),
        nn.ReLU(),
        nn.Linear(10, 1),
    )

def train_gd(X, y, epochs=80, batch_size=None, lr=0.05):
    """batch_size=None → full batch; 1 → SGD; else mini-batch."""
    model = make_yield_net()
    opt = optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    n = len(X)
    history = []
    for epoch in range(epochs):
        perm = torch.randperm(n)
        Xp, yp = X[perm], y[perm]
        bs = n if batch_size is None else batch_size
        epoch_loss = 0.0
        steps = 0
        for i in range(0, n, bs):
            xb, yb = Xp[i:i+bs], yp[i:i+bs]
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
            steps += 1
        history.append(epoch_loss / steps)
    model.eval()
    with torch.no_grad():
        val_mse = loss_fn(model(X_val), y_val).item()
    return history, val_mse, model

losses_batch, val_b, _ = train_gd(X_train, y_train, batch_size=None)
losses_mini_batch, val_m, _ = train_gd(X_train, y_train, batch_size=16)
losses_sgd, val_s, best_model = train_gd(X_train, y_train, batch_size=1)
print(f'val MSE  batch={val_b:.4f}  mini={val_m:.4f}  sgd={val_s:.4f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses_batch, label='Batch GD', marker='o', markevery=10)
plt.plot(losses_mini_batch, label='Mini-Batch GD', marker='x', markevery=10)
plt.plot(losses_sgd, label='Stochastic GD', marker='^', markevery=10)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Gradient Descent Strategies: Loss Convergence')
plt.legend()
plt.show()